# PyTorch Benchmark (No Transformers)


In [9]:
import os
import re
import sys
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

sys.path.append("..")

from src.data_utils import (
    build_label_maps,
    create_tfidf_features,
    encode_labels,
    make_dataloaders_from_arrays,
    stratified_split,
 )
from src.pytorch_models import build_model, select_device
from src.train_utils import evaluate, run_training, save_artifact, set_seed

In [10]:
SEED = 2026
VAL_RATIO = 0.2
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

MAX_TFIDF_FEATURES = 2000
MAX_VOCAB = 12000
MAX_LEN = 180
MIN_FREQ = 2

set_seed(SEED)
device = select_device()
print("Using device:", device)

Using device: cpu


In [11]:
df = pd.read_csv("../data/dataset_completo.csv")

text_col = "text" if "text" in df.columns else "Text"
label_col = "label" if "label" in df.columns else "Label"

df[text_col] = df[text_col].fillna("").astype(str)
df[label_col] = df[label_col].astype(str).str.strip()

class_order = ["Human", "Anthropic", "Google", "OpenAI", "Meta"]
present_classes = sorted(df[label_col].unique())
print("Classes in dataset:", present_classes)

missing = [c for c in class_order if c not in present_classes]
if missing:
    print("Warning: classes not present in current dataset:", missing)

df_train, df_val = stratified_split(
    df, text_col=text_col, label_col=label_col, test_size=VAL_RATIO, seed=SEED
)

print(f"Train size: {len(df_train)} | Val size: {len(df_val)}")
print("Train label distribution:")
print(df_train[label_col].value_counts())

Classes in dataset: ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
Train size: 481 | Val size: 121
Train label distribution:
label
Meta         97
OpenAI       96
Human        96
Anthropic    96
Google       96
Name: count, dtype: int64


In [12]:
# Fixed label maps for all models
label_to_idx, idx_to_label = build_label_maps(class_order)

y_train = encode_labels(df_train[label_col].values, label_to_idx)
y_val = encode_labels(df_val[label_col].values, label_to_idx)

valid_idx = [label_to_idx[c] for c in present_classes if c in label_to_idx]
print("Label indices used in metrics:", valid_idx)

Label indices used in metrics: [1, 2, 0, 4, 3]


In [13]:
def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def build_vocab(texts, max_vocab=12000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize_text(text))

    vocab = {"<pad>": 0, "<unk>": 1}
    for token, freq in counter.most_common():
        if len(vocab) >= max_vocab:
            break
        if freq < min_freq:
            continue
        vocab[token] = len(vocab)
    return vocab


def encode_text_to_ids(text, vocab, max_len=180):
    tokens = tokenize_text(text)
    ids = [vocab.get(t, 1) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return ids


def encode_texts_to_matrix(texts, vocab, max_len=180):
    return np.array([encode_text_to_ids(t, vocab, max_len) for t in texts], dtype=np.int64)

In [14]:
# Feature pipeline A: TF-IDF for mirror baseline
tfidf, X_train_tfidf, X_val_tfidf = create_tfidf_features(
    df_train[text_col].values, df_val[text_col].values, max_features=MAX_TFIDF_FEATURES
)

train_loader_tfidf, val_loader_tfidf = make_dataloaders_from_arrays(
    X_train_tfidf, y_train, X_val_tfidf, y_val, batch_size=BATCH_SIZE
)

print("TF-IDF shapes:", X_train_tfidf.shape, X_val_tfidf.shape)

# Feature pipeline B: Token IDs for embedding/recurrent models
vocab = build_vocab(df_train[text_col].values, max_vocab=MAX_VOCAB, min_freq=MIN_FREQ)
X_train_ids = encode_texts_to_matrix(df_train[text_col].values, vocab, max_len=MAX_LEN)
X_val_ids = encode_texts_to_matrix(df_val[text_col].values, vocab, max_len=MAX_LEN)

train_loader_ids, val_loader_ids = make_dataloaders_from_arrays(
    X_train_ids, y_train, X_val_ids, y_val, batch_size=BATCH_SIZE
)

print("Token-ID shapes:", X_train_ids.shape, X_val_ids.shape)
print("Vocab size:", len(vocab))

Vocabulário criado com 2000 palavras
TF-IDF shapes: (481, 2000) (121, 2000)
Token-ID shapes: (481, 180) (121, 180)
Vocab size: 3322


In [15]:
# Quick benchmark only (no hyperparameter optimization section)

EPOCHS_QUICK = 8
PATIENCE_QUICK = 3

model_specs = [
    (
        "mlp_tfidf",
        {
            "input_size": X_train_tfidf.shape[1],
            "hidden_size": 256,
            "num_classes": len(class_order),
            "dropout": 0.3,
        },
        train_loader_tfidf,
        val_loader_tfidf,
        LR,
    ),
    (
        "embed_avg",
        {
            "vocab_size": len(vocab),
            "embed_dim": 128,
            "hidden_dim": 128,
            "num_classes": len(class_order),
            "padding_idx": 0,
            "dropout": 0.3,
        },
        train_loader_ids,
        val_loader_ids,
        LR,
    ),
    (
        "rnn",
        {
            "vocab_size": len(vocab),
            "embed_dim": 128,
            "hidden_dim": 128,
            "num_layers": 1,
            "num_classes": len(class_order),
            "padding_idx": 0,
            "dropout": 0.2,
        },
        train_loader_ids,
        val_loader_ids,
        LR,
    ),
    (
        "lstm",
        {
            "vocab_size": len(vocab),
            "embed_dim": 128,
            "hidden_dim": 128,
            "num_layers": 1,
            "num_classes": len(class_order),
            "padding_idx": 0,
            "dropout": 0.2,
        },
        train_loader_ids,
        val_loader_ids,
        LR,
    ),
    (
        "gru",
        {
            "vocab_size": len(vocab),
            "embed_dim": 128,
            "hidden_dim": 128,
            "num_layers": 1,
            "num_classes": len(class_order),
            "padding_idx": 0,
            "dropout": 0.2,
        },
        train_loader_ids,
        val_loader_ids,
        LR,
    ),
]

results = []
artifacts = {}

for model_name, model_cfg, train_loader, val_loader, lr_value in model_specs:
    print("\n" + "=" * 70)
    print(f"Benchmark: {model_name}")

    model = build_model(model_name, model_cfg).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr_value)

    history, best_state, best_epoch = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS_QUICK,
        patience=PATIENCE_QUICK,
        min_delta=1e-4,
        verbose=True,
    )

    model.load_state_dict(best_state)
    val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader, criterion, device)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    run_key = f"{model_name}_quick"
    row = {
        "run_key": run_key,
        "model": model_name,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": macro_f1,
        "best_epoch": best_epoch,
        "epochs": EPOCHS_QUICK,
        "lr": lr_value,
    }
    results.append(row)

    artifacts[run_key] = {
        "model_name": model_name,
        "model_config": model_cfg,
        "state_dict": best_state,
        "class_order": class_order,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "history": history,
        "stage": "quick",
    }

final_results_df = pd.DataFrame(results).sort_values(
    ["val_macro_f1", "val_acc"], ascending=False
).reset_index(drop=True)

print("\nBenchmark ranking (quick only):")
display(final_results_df)


Benchmark: mlp_tfidf
Epoch 01/8 | train_loss=1.6053 train_acc=0.2890 | val_loss=1.5942 val_acc=0.3884
Epoch 02/8 | train_loss=1.5814 train_acc=0.4407 | val_loss=1.5679 val_acc=0.5950
Epoch 03/8 | train_loss=1.5456 train_acc=0.6112 | val_loss=1.5304 val_acc=0.6033
Epoch 04/8 | train_loss=1.4983 train_acc=0.6798 | val_loss=1.4790 val_acc=0.6612
Epoch 05/8 | train_loss=1.4319 train_acc=0.7900 | val_loss=1.4152 val_acc=0.7190
Epoch 06/8 | train_loss=1.3574 train_acc=0.8732 | val_loss=1.3402 val_acc=0.8182
Epoch 07/8 | train_loss=1.2705 train_acc=0.9064 | val_loss=1.2616 val_acc=0.8430
Epoch 08/8 | train_loss=1.1796 train_acc=0.9356 | val_loss=1.1823 val_acc=0.8512

Benchmark: embed_avg
Epoch 01/8 | train_loss=1.5941 train_acc=0.3285 | val_loss=1.5523 val_acc=0.5124
Epoch 02/8 | train_loss=1.5370 train_acc=0.5364 | val_loss=1.4759 val_acc=0.5702
Epoch 03/8 | train_loss=1.4681 train_acc=0.6299 | val_loss=1.3839 val_acc=0.6116
Epoch 04/8 | train_loss=1.3707 train_acc=0.6778 | val_loss=1.2671

,run_key,model,val_loss,val_acc,val_macro_f1,best_epoch,epochs,lr
0,mlp_tfidf_quick,mlp_tfidf,1.182339,0.851240,0.832184,8,8,0.001
1,embed_avg_quick,embed_avg,0.771333,0.776860,0.766670,8,8,0.001
2,rnn_quick,rnn,1.610122,0.206612,0.068493,1,8,0.001
3,lstm_quick,lstm,1.609324,0.206612,0.068493,4,8,0.001
4,gru_quick,gru,1.609382,0.206612,0.068493,2,8,0.001


In [8]:
best_run_key = final_results_df.iloc[0]["run_key"]
best_payload = artifacts[best_run_key].copy()

if best_payload["model_name"] == "mlp_tfidf":
    best_payload["preprocess_type"] = "tfidf"
    best_payload["tfidf"] = tfidf
else:
    best_payload["preprocess_type"] = "token_ids"
    best_payload["vocab"] = vocab
    best_payload["max_len"] = MAX_LEN

os.makedirs("../modelos", exist_ok=True)
best_artifact_path = f"../modelos/modelo_B_{best_payload['model_name']}_{best_run_key}.pt"
save_artifact(best_artifact_path, best_payload)

print("Saved best artifact:", best_artifact_path)
print("Best run:", best_run_key)
print("Best model:", best_payload["model_name"])

Saved best artifact: ../modelos/modelo_B_hierarchical_gru_hierarchical_gru_trial_1.pt
Best run: hierarchical_gru_trial_1
Best model: hierarchical_gru
